# Topic 02 — Telecom Churn Visual EDA

A visual mini-research project on the official `telecom_churn.csv` dataset from mlcourse.ai.

The structure is intentionally explicit: **question → hypothesis → visualization → numerical check → conclusion**. This is the reasoning I want to preserve in the repository.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_URL = 'https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/'
df = pd.read_csv(DATA_URL + 'telecom_churn.csv')
print(df.shape)
df.head()

## 1. Start with the target

**Question.** How imbalanced is churn?

**Why first.** Before comparing features, I need to know the base rate. Otherwise a visually large subgroup can be misleading simply because one class dominates the dataset.

In [ ]:
target_share = df['Churn'].value_counts(normalize=True).sort_index()
print(target_share)
sns.countplot(data=df, x='Churn')
plt.title('Customer churn counts')
plt.show()

**Takeaway.** Churn is a minority class. Later, when I compare categories, proportions inside each category are often more meaningful than raw counts.

## 2. International plan as a churn hypothesis

**Hypothesis.** Customers with an international plan may churn more often.

A grouped countplot is useful for orientation, but the decisive check is the churn rate inside each plan group.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='International plan', hue='Churn', ax=ax)
ax.set_title('Churn by international plan')
plt.show()

plan_rate = pd.crosstab(df['International plan'], df['Churn'], normalize='index')
plan_rate

**Interpretation.** If the churn share for `Yes` is much higher, the feature is a plausible predictor. This is an association, not proof that the plan causes churn.

## 3. Customer service calls

**Hypothesis.** Repeated calls to customer support may signal unresolved problems and therefore higher churn risk.

Because the feature is a small-count integer, a countplot split by target is more readable than a continuous density plot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.countplot(data=df, x='Customer service calls', hue='Churn', ax=ax)
ax.set_title('Customer service calls split by churn')
plt.show()

service_rate = df.groupby('Customer service calls')['Churn'].agg(['mean','count'])
service_rate

**What I check next.** A jump in churn rate after several support calls is more actionable than a vague average difference. I also inspect `count`: extreme call counts may contain too few clients to support a stable rate.

## 4. Daytime usage

**Question.** Is high daytime usage associated with churn?

I compare the full distributions instead of only the means because two groups can have similar averages and different tails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x='Total day minutes', hue='Churn', kde=True, element='step', stat='density', common_norm=False, ax=axes[0])
sns.boxplot(data=df, x='Churn', y='Total day minutes', ax=axes[1])
axes[0].set_title('Day minutes distributions')
axes[1].set_title('Day minutes by churn')
plt.tight_layout()
plt.show()
df.groupby('Churn')['Total day minutes'].agg(['mean','median','std'])

## 5. Redundant usage variables

**Question.** Do minutes and charges duplicate the same signal?

If charge is almost a deterministic transformation of minutes, keeping both can add little information and can distort interpretations of feature importance later.

In [ ]:
pairs = [
    ('Total day minutes','Total day charge'),
    ('Total eve minutes','Total eve charge'),
    ('Total night minutes','Total night charge'),
    ('Total intl minutes','Total intl charge'),
]
for minutes, charge in pairs:
    print(minutes, '<->', charge, round(df[minutes].corr(df[charge]), 6))

sample = df.sample(1000, random_state=42)
sns.scatterplot(data=sample, x='Total day minutes', y='Total day charge', hue='Churn', alpha=.6)
plt.title('Day minutes and charge are nearly deterministic')
plt.show()

## 6. Correlation screening

**Question.** Which numerical variables deserve a second look because they move together strongly?

I use a heatmap for screening, then interpret individual pairs. A heatmap is not a feature-selection algorithm by itself.

In [ ]:
numeric = df.select_dtypes(include='number')
corr = numeric.corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Numerical feature correlations')
plt.tight_layout()
plt.show()

## 7. Compact feature hypotheses

At this stage I would carry these hypotheses into later modeling:

1. `International plan` is likely informative because churn proportions differ by plan status.
2. `Customer service calls` may contain a threshold-like relationship with churn.
3. `Total day minutes` appears to separate churn groups better than many raw count features.
4. Minute/charge pairs are strongly redundant and should be treated carefully.

These are **modeling hypotheses**, not final truths. Topic 03+ should test whether they improve out-of-sample prediction.

## Final checklist

- Did I formulate a question before drawing each chart? **Yes.**
- Did I verify important visual impressions numerically? **Yes.**
- Did I avoid turning association into causation? **Yes.**
- Did I produce hypotheses that can be tested later? **Yes.**